# Model Comparison — which model should the app use?

We train **7 different models** (chosen from our reference papers) with **exactly the same data, split, augmentation, epochs and settings**,
so the only difference is the model architecture. Then we compare them on what matters for **our app**
(Triage + Patient explainer) and pick the best one.

**Kaggle settings (right panel):** Add Input → `brain-tumor-mri-dataset` · Accelerator → GPU T4 · Internet → On.

**Tip:** use **Save Version → Save & Run All** (top right). It runs in the background,
so you can close the browser (about 1–1.5 hours). Then open the version's **Output** tab to download the files.

## Our selection rule (decided BEFORE seeing results)

We write the rule first, so we don't choose the model that just "looks best" afterwards.

1. **Must be fast enough for free hosting**: under 1 second per image (prediction + heatmap) on 2 CPU cores
   (Hugging Face free Spaces have 2 CPUs, no GPU).
2. **Fewest silent missed tumors**: a tumor predicted as "no tumor" **and not flagged** by the safety rule.
   This is the most dangerous error for triage — nobody would check that scan.
3. If tied: **highest macro-F1** (balanced accuracy across all 4 classes).
4. If still tied: **faster / smaller** model.

In [ ]:
!pip install -q grad-cam timm

In [ ]:
import os, random, time, copy, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## Settings
The same for every model. Only `MODELS` changes.

In [ ]:
MODELS = [
    # chosen from our 13 reference papers (7 of them use this same Kaggle dataset)
    "resnet50",          # most common baseline (Ali 2026, Yoon 2025, Afnaan 2025; ResNet18/101 in Das 2025, Thanon 2026)
    "vgg16",             # classic baseline (Yoon 2025, Thanon 2026, Afnaan 2025, Aiya 2025) — very big, expected too slow for free hosting
    "efficientnet_b0",   # our first model (Ali 2026, Yoon 2025, Shakir 2026)
    "densenet121",       # popular in medical imaging (Ali 2026, Shakir 2026, Afnaan 2025)
    "legacy_xception",   # Xception (Shakir 2026, Gharaibeh 2025)
    "convnext_tiny",     # modern CNN (Shakir 2026)
    "mobilenetv2_100",   # lightweight model (Afnaan 2025), fast for free hosting
]
SEED, IMG_SIZE, BATCH_SIZE, EPOCHS, LR, VAL_RATIO = 42, 224, 32, 12, 1e-4, 0.15
PRETRAINED = True
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
OUT_DIR = Path("/kaggle/working"); INPUT_ROOT = Path("/kaggle/input")
(OUT_DIR / "models").mkdir(parents=True, exist_ok=True)

# Safety rule — same numbers as core/predictor.py in the app
MIN_CONFIDENCE, MIN_MARGIN, NOTUMOR_TRUST = 0.80, 0.20, 0.95
CPU_THREADS = 2          # like a free Hugging Face Space
MAX_CPU_SECONDS = 1.0    # rule 1

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

## Step 1 — Load the data once (into memory)
We resize every image to 224×224 **once** and keep it in RAM. This makes training 6 models much faster,
because we don't read and resize the files again for every epoch.

In [ ]:
def find_data_dir(root):
    for p in sorted(root.rglob("Training")):
        if p.is_dir() and (p.parent / "Testing").is_dir():
            return p.parent
    raise FileNotFoundError("Training/ + Testing/ not found. Did you add the dataset?")

DATA_DIR = find_data_dir(INPUT_ROOT)
IMG_EXT = {".jpg", ".jpeg", ".png"}
def list_images(d):
    return pd.DataFrame([{"path": str(f), "label": c.name}
                         for c in sorted(p for p in d.iterdir() if p.is_dir())
                         for f in c.iterdir() if f.suffix.lower() in IMG_EXT])

train_full_df = list_images(DATA_DIR / "Training")
test_df = list_images(DATA_DIR / "Testing")
CLASS_NAMES = sorted(train_full_df.label.unique())
C2I = {c: i for i, c in enumerate(CLASS_NAMES)}
NT = C2I["notumor"]

train_df, val_df = train_test_split(train_full_df, test_size=VAL_RATIO,
                                    stratify=train_full_df.label, random_state=SEED)

def load_all(df):
    return [Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)) for p in df.path]

t0 = time.time()
train_imgs, val_imgs, test_imgs = load_all(train_df), load_all(val_df), load_all(test_df)
print(f"train {len(train_imgs)} | val {len(val_imgs)} | test {len(test_imgs)} | loaded in {time.time()-t0:.0f}s")

## Step 2 — Same preprocessing for every model

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class MemDataset(Dataset):
    def __init__(self, imgs, labels, tf):
        self.imgs, self.labels, self.tf = imgs, [C2I[l] for l in labels], tf
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i): return self.tf(self.imgs[i]), self.labels[i]

def loader(imgs, df, tf, shuffle, seed=SEED):
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(MemDataset(imgs, df.label, tf), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=2, generator=g)

val_loader = loader(val_imgs, val_df, eval_tf, False)
test_loader = loader(test_imgs, test_df, eval_tf, False)
test_y = torch.tensor([C2I[l] for l in test_df.label])

## Step 3 — Helper functions
- `train_model`: same training loop as before (AdamW, cosine schedule, mixed precision, keep best validation epoch)
- `fit_temperature`: temperature scaling (honest confidence)
- `evaluate`: all the numbers we compare
- `cpu_speed`: time for one prediction + heatmap on 2 CPU cores (like free hosting)

In [ ]:
def cam_layer_name(model):
    # the last feature stage of the network; works for all timm CNNs we use
    return model.feature_info[-1]["module"]

def train_model(name):
    set_seed(SEED)
    model = timm.create_model(name, pretrained=PRETRAINED, num_classes=len(CLASS_NAMES)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    use_amp = device == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    tr_loader = loader(train_imgs, train_df, train_tf, True)
    best_acc, best_state, hist = 0, None, []
    t0 = time.time()
    for ep in range(1, EPOCHS + 1):
        model.train()
        for x, y in tr_loader:
            x, y = x.to(device), y.to(device)
            with torch.autocast("cuda", enabled=use_amp):
                loss = F.cross_entropy(model(x), y)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        v_logits, v_y = get_logits(model, val_loader)
        acc = (v_logits.argmax(1) == v_y).float().mean().item()
        hist.append(acc)
        if acc > best_acc:
            best_acc, best_state = acc, copy.deepcopy(model.state_dict())
        print(f"  {name} epoch {ep:2d}/{EPOCHS} val acc {acc:.4f}{'  <- best' if acc == best_acc else ''}")
    model.load_state_dict(best_state)
    return model, best_acc, hist, time.time() - t0

@torch.no_grad()
def get_logits(model, dl):
    model.eval(); L, Y = [], []
    for x, y in dl:
        L.append(model(x.to(device)).float().cpu()); Y.append(y)
    return torch.cat(L), torch.cat(Y)

def fit_temperature(logits, y):
    log_T = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_T], lr=0.1, max_iter=200)
    def closure():
        opt.zero_grad(); l = F.cross_entropy(logits / log_T.exp(), y); l.backward(); return l
    opt.step(closure)
    return log_T.exp().item()

def ece(probs, y, n_bins=15):
    conf, pred = probs.max(1); correct = pred.eq(y).float(); e = 0.0
    for lo, hi in zip(torch.linspace(0, 1, n_bins + 1)[:-1], torch.linspace(0, 1, n_bins + 1)[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.float().mean() * (conf[m].mean() - correct[m].mean()).abs()
    return float(e)

def evaluate(logits, T):
    probs = F.softmax(logits / T, 1)
    conf, pred = probs.max(1)
    top2 = probs.topk(2, 1).values
    refer = (conf < MIN_CONFIDENCE) | ((top2[:, 0] - top2[:, 1]) < MIN_MARGIN) | ((pred == NT) & (probs[:, NT] < NOTUMOR_TRUST))
    wrong = pred != test_y
    is_tumor = test_y != NT
    missed = is_tumor & (pred == NT)                  # tumor called "no tumor"
    false_alarm = (~is_tumor) & (pred != NT)          # normal called "tumor"
    recalls = recall_score(test_y, pred, average=None, labels=range(len(CLASS_NAMES)))
    return {
        "test_acc": accuracy_score(test_y, pred),
        "macro_f1": f1_score(test_y, pred, average="macro"),
        **{f"recall_{c}": r for c, r in zip(CLASS_NAMES, recalls)},
        "tumor_sensitivity": 1 - missed.sum().item() / is_tumor.sum().item(),
        "missed_tumors": missed.sum().item(),
        "silent_missed_tumors": (missed & ~refer).sum().item(),
        "false_alarms": false_alarm.sum().item(),
        "ece": ece(probs, test_y),
        "flagged_pct": refer.float().mean().item(),
        "errors_flagged_pct": (wrong & refer).sum().item() / max(wrong.sum().item(), 1),
        "acc_unflagged": (~wrong[~refer]).float().mean().item(),
    }, pred

def cpu_speed(model, layer, n=10):
    torch.set_num_threads(CPU_THREADS)
    m = copy.deepcopy(model).cpu().eval()
    cam = GradCAM(model=m, target_layers=[m.get_submodule(layer)])
    x = eval_tf(test_imgs[0]).unsqueeze(0)
    with torch.no_grad(): m(x)  # warm-up
    t = time.time()
    for _ in range(n):
        with torch.no_grad(): idx = int(m(x).argmax())
        cam(input_tensor=x, targets=[ClassifierOutputTarget(idx)])
    torch.set_num_threads(os.cpu_count())
    return (time.time() - t) / n

## Step 4 — Train and evaluate all models
This is the long cell (about 1–1.5 hours on a T4; VGG16 is the slowest). Each model is saved to `/kaggle/working/models/`.

In [ ]:
results, preds, histories = [], {}, {}
for name in MODELS:
    print(f"\n===== {name} =====")
    model, best_val, hist, train_sec = train_model(name)
    histories[name] = hist
    layer = cam_layer_name(model)
    v_logits, v_y = get_logits(model, val_loader)
    T = fit_temperature(v_logits, v_y)
    t_logits, _ = get_logits(model, test_loader)
    metrics, pred = evaluate(t_logits, T)
    preds[name] = pred
    path = OUT_DIR / "models" / f"{name}.pth"
    torch.save({"model_name": name, "state_dict": model.state_dict(), "class_names": CLASS_NAMES,
                "img_size": IMG_SIZE, "mean": MEAN, "std": STD, "temperature": T,
                "cam_layer": layer, "test_accuracy": float(metrics["test_acc"])}, path)
    row = {"model": name, "params_M": sum(p.numel() for p in model.parameters()) / 1e6,
           "file_MB": path.stat().st_size / 1e6, "train_min": train_sec / 60,
           "best_val_acc": best_val, "temperature": T, "cam_layer": layer,
           "cpu_sec_per_image": cpu_speed(model, layer), **metrics}
    results.append(row)
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})
    del model; torch.cuda.empty_cache()

res = pd.DataFrame(results)
res.to_csv(OUT_DIR / "model_comparison.csv", index=False)

## Step 5 — Comparison table

In [ ]:
show = res[["model", "test_acc", "macro_f1", "recall_glioma", "tumor_sensitivity", "missed_tumors",
            "silent_missed_tumors", "false_alarms", "ece", "flagged_pct", "errors_flagged_pct",
            "params_M", "file_MB", "cpu_sec_per_image", "train_min"]].copy()
pct = ["test_acc", "macro_f1", "recall_glioma", "tumor_sensitivity", "flagged_pct", "errors_flagged_pct"]
display(show.style.format({**{c: "{:.2%}" for c in pct}, "ece": "{:.3f}", "params_M": "{:.1f}",
                           "file_MB": "{:.1f}", "cpu_sec_per_image": "{:.2f}", "train_min": "{:.1f}"})
        .highlight_max(subset=["test_acc", "macro_f1", "recall_glioma", "tumor_sensitivity"], color="#c6efce")
        .highlight_min(subset=["missed_tumors", "silent_missed_tumors", "ece", "cpu_sec_per_image"], color="#c6efce"))

**How to read the columns**
- `test_acc` / `macro_f1`: overall quality (macro-F1 treats all 4 classes equally).
- `recall_glioma`: our weakest class before (80.7%).
- `tumor_sensitivity`: of all real tumors, how many were called *some* tumor (not "no tumor").
- `missed_tumors`: tumors called "no tumor". `silent_missed_tumors`: of those, how many the safety rule did **not** flag.
- `false_alarms`: normal scans called "tumor" (less dangerous: a doctor checks them).
- `ece`: calibration error after temperature scaling (lower = more honest confidence).
- `flagged_pct`: how many scans the rule sends to a doctor (too high = extra work).
- `errors_flagged_pct`: of all wrong answers, how many the rule caught.
- `cpu_sec_per_image`: prediction + heatmap on 2 CPU cores (free hosting speed).

## Step 6 — Charts

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
order = res.sort_values("macro_f1", ascending=False)
ax[0].barh(order.model, order.macro_f1); ax[0].set_xlim(order.macro_f1.min() - 0.03, 1); ax[0].set_title("Macro-F1 (higher is better)")
ax[1].barh(order.model, order.missed_tumors, label="missed tumors")
ax[1].barh(order.model, order.silent_missed_tumors, label="not flagged (silent)")
ax[1].set_title("Missed tumors (lower is better)"); ax[1].legend()
ax[2].scatter(res.cpu_sec_per_image, res.macro_f1, s=res.params_M * 15)
for _, r in res.iterrows(): ax[2].annotate(r.model, (r.cpu_sec_per_image, r.macro_f1), fontsize=8)
ax[2].set_xlabel("CPU seconds per image (2 cores)"); ax[2].set_ylabel("Macro-F1"); ax[2].set_title("Quality vs speed (bubble = size)")
plt.tight_layout(); plt.savefig(OUT_DIR / "comparison_charts.png", dpi=150); plt.show()

plt.figure(figsize=(7, 4))
for n, h in histories.items(): plt.plot(range(1, len(h) + 1), h, marker=".", label=n)
plt.xlabel("epoch"); plt.ylabel("validation accuracy"); plt.legend(fontsize=8); plt.title("Validation accuracy per epoch")
plt.tight_layout(); plt.savefig(OUT_DIR / "val_curves.png", dpi=150); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for a, name in zip(axes.flat, MODELS):
    sns.heatmap(confusion_matrix(test_y, preds[name]), annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=a)
    a.set_title(name); a.set_xlabel("Predicted"); a.set_ylabel("True")
for a in axes.flat[len(MODELS):]: a.axis("off")
plt.tight_layout(); plt.savefig(OUT_DIR / "confusion_matrices.png", dpi=150); plt.show()

## Step 7 — Apply the selection rule

In [ ]:
ok = res[res.cpu_sec_per_image <= MAX_CPU_SECONDS].copy()
too_slow = sorted(set(res.model) - set(ok.model))
if too_slow: print("Removed (too slow for free hosting):", too_slow)
ranked = ok.sort_values(["silent_missed_tumors", "macro_f1", "cpu_sec_per_image"], ascending=[True, False, True])
display(ranked[["model", "silent_missed_tumors", "missed_tumors", "macro_f1", "test_acc", "cpu_sec_per_image"]])
BEST = ranked.iloc[0].model
print(f"\nSelected model: {BEST}")

## Step 8 — Heatmaps of all models on the same images
Useful for the presentation: do the models look at the tumor? Do they agree?

In [ ]:
def overlay(name, img):
    ck = torch.load(OUT_DIR / "models" / f"{name}.pth", map_location="cpu", weights_only=False)
    m = timm.create_model(name, pretrained=False, num_classes=len(CLASS_NAMES))
    m.load_state_dict(ck["state_dict"]); m.eval()
    cam = GradCAM(model=m, target_layers=[m.get_submodule(ck["cam_layer"])])
    x = eval_tf(img).unsqueeze(0)
    with torch.no_grad(): p = F.softmax(m(x) / ck["temperature"], 1)[0]
    idx = int(p.argmax())
    heat = cam(input_tensor=x, targets=[ClassifierOutputTarget(idx)])[0]
    return show_cam_on_image(np.asarray(img, dtype=np.float32) / 255, heat, use_rgb=True), CLASS_NAMES[idx], float(p[idx])

rng = np.random.default_rng(0)
picks = [int(rng.choice(np.where(test_y.numpy() == C2I[c])[0])) for c in ["glioma", "meningioma", "pituitary"]]
fig, axes = plt.subplots(len(picks), len(MODELS) + 1, figsize=(2.6 * (len(MODELS) + 1), 2.9 * len(picks)))
for r, i in enumerate(picks):
    axes[r, 0].imshow(test_imgs[i]); axes[r, 0].set_title(f"true: {test_df.label.iloc[i]}", fontsize=9)
    for c, name in enumerate(MODELS, start=1):
        im, lab, conf = overlay(name, test_imgs[i])
        axes[r, c].imshow(im); axes[r, c].set_title(f"{name}\n{lab} {conf:.0%}", fontsize=8)
for a in axes.flat: a.axis("off")
plt.tight_layout(); plt.savefig(OUT_DIR / "gradcam_comparison.png", dpi=150); plt.show()

## Step 9 — Save the winner for the app
Download these from the **Output** panel:
- `best_model.pth` → put it in the project's `model/` folder (replace the old one)
- `model_comparison.csv`, `comparison_charts.png`, `confusion_matrices.png`, `gradcam_comparison.png`, `val_curves.png` → for the report and slides

In [ ]:
import shutil
shutil.copy(OUT_DIR / "models" / f"{BEST}.pth", OUT_DIR / "best_model.pth")
ck = torch.load(OUT_DIR / "best_model.pth", map_location="cpu", weights_only=False)
print("best_model.pth =", ck["model_name"], "| cam_layer:", ck["cam_layer"], "| T =", round(ck["temperature"], 3),
      "| test acc:", round(ck["test_accuracy"], 4))

## Limitations of this comparison (write these in the report)
- **One run per model (one random seed).** Small differences (for example 0.3% accuracy) may be random noise.
  If two models are very close, re-run both with another `SEED` to check.
- **Same settings for all models.** Some models might do better with their own learning rate or more epochs.
  We chose equal settings to make the comparison fair and simple.
- **Speed was measured on Kaggle's CPU**, which is similar to (but not the same as) the hosting CPU.